# Week 4 - Module 4: Temporal Vital Forecasting
### BiLSTM over rolling ICU vitals windows (Heart Rate, SpO2)

Real data source: **MIMIC-III/IV ICU vitals** (credentialing pending per README Dataset Strategy). Falls back automatically to a synthetic-but-physiologically-shaped vitals generator if `mimic_dir` isn't set or the folder isn't found, so the notebook never gets blocked by a pending approval.

Run all cells top to bottom.

In [ ]:
import os
import sys
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Cardiac_Risk_Prediction'
except ImportError:
    PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.append(f'{PROJECT_DIR}/src')

print(f'Working directory permanently mapped to: {PROJECT_DIR}')


In [ ]:
import sys
sys.path.append(f'{PROJECT_DIR}/src')

from vitals_data import load_vitals_sequences, SEQ_LEN, FORECAST_HORIZON
from vitals_models import VitalsBiLSTM

## Load vitals sequences
Set `MIMIC_DIR` to a MIMIC-III/IV demo folder under `data/` if you've uploaded one (e.g. `mimic-iv-clinical-database-demo-2.2`). Leave as `None` to use the synthetic fallback - it still trains a real model, just on physiologically-shaped synthetic data rather than real patients.

In [ ]:
MIMIC_DIR = None  # e.g. f'{PROJECT_DIR}/data/mimic-iv-clinical-database-demo-2.2'

X, y, labels = load_vitals_sequences(mimic_dir=MIMIC_DIR)
print(f"Sequences: {len(X)} | X shape: {X.shape} | y (forecast target) shape: {y.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visual sanity check: plot one input window + its true forecast continuation
idx = 0
full_hr = np.concatenate([X[idx, :, 0], y[idx, :, 0]])
full_spo2 = np.concatenate([X[idx, :, 1], y[idx, :, 1]])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(full_hr); axes[0].axvline(SEQ_LEN, color='red', linestyle='--', label='forecast starts')
axes[0].set_title('Heart Rate: input window + true continuation'); axes[0].legend()
axes[1].plot(full_spo2); axes[1].axvline(SEQ_LEN, color='red', linestyle='--', label='forecast starts')
axes[1].set_title('SpO2: input window + true continuation'); axes[1].legend()
plt.tight_layout()

## Split, standardize, and build DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize per-feature (HR, SpO2 have very different scales) using
# stats from X_train only, applied identically to X_val and both y sets.
n_features = X_train.shape[-1]
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, n_features))

def scale(arr):
    shape = arr.shape
    return scaler.transform(arr.reshape(-1, n_features)).reshape(shape)

X_train_s, X_val_s = scale(X_train), scale(X_val)
y_train_s, y_val_s = scale(y_train), scale(y_val)

train_ds = TensorDataset(torch.tensor(X_train_s, dtype=torch.float32),
                          torch.tensor(y_train_s, dtype=torch.float32))
val_ds = TensorDataset(torch.tensor(X_val_s, dtype=torch.float32),
                        torch.tensor(y_val_s, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

## Build, train (or load checkpoint), and validate VitalsBiLSTM

In [ ]:
import torch.nn as nn
import torch.optim as optim

CHECKPOINT_PATH = f'{PROJECT_DIR}/checkpoints/module4_bilstm.pt'

model = VitalsBiLSTM(n_features=n_features, horizon=FORECAST_HORIZON).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

if os.path.exists(CHECKPOINT_PATH):
    print("Found pre-trained checkpoint in Drive! Loading instead of retraining.")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
else:
    print("No checkpoint found - training from scratch.")
    EPOCHS = 60
    train_losses, val_losses = [], []
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
        train_losses.append(running_loss / len(train_ds))

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = model(batch_x)
                val_running += criterion(preds, batch_y).item() * batch_x.size(0)
        val_losses.append(val_running / len(val_ds))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - train MSE: {train_losses[-1]:.4f} - val MSE: {val_losses[-1]:.4f}")

    torch.save(model.state_dict(), CHECKPOINT_PATH)
    print(f"Saved trained model to {CHECKPOINT_PATH}")

    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label='train MSE')
    plt.plot(val_losses, label='val MSE')
    plt.title('Module 4 - BiLSTM Forecasting Loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE (standardized scale)'); plt.legend()
    plt.savefig(f'{PROJECT_DIR}/outputs/week4_bilstm_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

## Final validation metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(batch_y.numpy())

all_preds = np.concatenate(all_preds).reshape(-1, n_features)
all_targets = np.concatenate(all_targets).reshape(-1, n_features)

mae = mean_absolute_error(all_targets, all_preds)
rmse = mean_squared_error(all_targets, all_preds) ** 0.5

print("=" * 45)
print("MODULE 4 - VITALS FORECASTING METRICS (standardized scale)")
print("=" * 45)
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

In [ ]:
# Visual: one example forecast vs actual (inverse-transformed back to real units)
sample_x, sample_y = X_val_s[0:1], y_val_s[0:1]
with torch.no_grad():
    pred = model(torch.tensor(sample_x, dtype=torch.float32).to(device)).cpu().numpy()

pred_real = scaler.inverse_transform(pred.reshape(-1, n_features)).reshape(pred.shape)
true_real = scaler.inverse_transform(sample_y.reshape(-1, n_features)).reshape(sample_y.shape)
input_real = scaler.inverse_transform(sample_x.reshape(-1, n_features)).reshape(sample_x.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for i, name in enumerate(['Heart Rate', 'SpO2']):
    axes[i].plot(range(SEQ_LEN), input_real[0, :, i], label='input window')
    axes[i].plot(range(SEQ_LEN, SEQ_LEN + FORECAST_HORIZON), true_real[0, :, i], label='true forecast', marker='o')
    axes[i].plot(range(SEQ_LEN, SEQ_LEN + FORECAST_HORIZON), pred_real[0, :, i], label='predicted forecast', marker='x')
    axes[i].set_title(name); axes[i].legend()
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/outputs/week4_forecast_example.png', dpi=150, bbox_inches='tight')
plt.show()

## Notes for the progress report
- Confirm from the printed message above whether this run used **real MIMIC data** or the **synthetic fallback** - report that honestly.
- The `pooled` feature vector returned by `model(x, return_features=True)` is what Week 6's fusion layer will consume from this branch, same pattern as `ECG1DCNN`.
- This module shares `vitals_data.py` with Module 5 (`vae_models.py`) - same real-data-first/synthetic-fallback loader, same input shape contract.